# Spatial variation of 3-D genome features across a tissue

Spatial Hi-C-RNA (Guo et al. 2026, *Cell*) measures contacts and the transcriptome in the same 20-µm spots of a
coronal mouse-brain section.  This tutorial asks of each spot's **global 3-D genome features** — the share of
short- and long-range contacts, the decay of contacts with distance, the cis fraction — whether they vary across
the tissue, and whether they vary **beyond what the spots' transcriptomes say**:

1. per-spot features from the contact-distance histograms stored in the atlas store (`uchrom.fea.contact_features`);
2. the tissue: maps, anatomical regions from marker genes, the paper's short / long-range pattern;
3. the barcode stripes of the microfluidic array and their removal (`uchrom.spatial.destripe`);
4. spatial tests — Moran's I and SPARK-X — raw and adjusted for depth and for the spot's RNA (`uchrom.spatial`).

**Data**: atlas store `hicrna_mouse_brain` (Guo et al. 2026, GEO GSE311199; 2 sections, 12,474 spots), opened
over HTTP — only the cells table, the positions and the per-spot histograms are read.  The histograms come from
one pass over the published pairs (`apps/atlas/recipes/spatial_hicrna/sherlock/contact_stats.sbatch`).
**Runtime**: a few minutes.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import uchrom.datasets as ds
from uchrom.fea import ContactFeatureParams, contact_features
from uchrom.pl import plot_spatial
from uchrom.spatial import destripe, smooth_share, sparkx, spatial_autocorr, spatial_neighbors

plt.rcParams["figure.dpi"] = 90
cd = ds.load("hicrna_mouse_brain")
print(cd)
print({k: v.shape for k, v in cd.cellm.items() if k.startswith("contacts")})
print(cd.uns["contact_stats"]["contacts"]["source"])

## 1. Global features of every spot

`contact_features` reads `cellm["contacts.distance"]` — per spot the cis contacts in log2 distance bins from
2.5 kb (8 per octave, the bins of scHiCluster) — and `contacts.cis` / `contacts.trans`.  With
`ContactFeatureParams.guo2026()` the short / long-range ratio is the paper's: contacts under 5 kb left out,
short = 207 kb – 1.97 Mb, long = 18.8 – 53.1 Mb.  The features are written to `cd.cells` as `contacts.*`.

In [ ]:
t = time.time()
f = contact_features(cd, params=ContactFeatureParams.guo2026())
print(f"{len(f):,} spots in {time.time() - t:.1f} s")
f[["n_cis", "cis_fraction", "frac_short", "frac_long", "short_long_ratio", "frac_2M_12M",
   "ps_slope_100k_1M", "ps_slope_1M_10M"]].describe().round(3)

Spots on the first and last barcode row carry several times the median contacts, one column of section r2 is
nearly empty (a dead barcode channel), and some spots are shallow: the QC of `benchmarks/spatial_composition`
keeps spots with at least 50,000 contacts off the edge rows and dead columns.

In [ ]:
c = cd.cells.copy()
c.index = [str(i) for i in cd._cell_ids_axis()]
c = c.join(f)
c["log_contacts"] = np.log10(c["hic_contacts"])
qc = np.zeros(len(c), bool)
for sec, g in c.groupby("section", observed=True):
    i = c.index.get_indexer(g.index)
    y, x, n = g.centroid_y.to_numpy(), g.centroid_x.to_numpy(), g.hic_contacts.to_numpy()
    col_median = pd.Series(n).groupby(x).median()
    dead = np.isin(x, col_median.index[col_median < 0.05 * np.median(n)])
    qc[i] = (n >= 50_000) & (y != y.min()) & (y != y.max()) & ~dead
c["qc"] = qc
print(c.groupby("section", observed=True)["qc"].agg(["sum", "size"]))

## 2. The tissue

The short / long-range ratio next to two marker genes (log1p CP10k of the spot's RNA): white matter (*Mbp*) and
the dentate gyrus (*Prox1*).

In [ ]:
q = c[c.qc]
fig, axes = plt.subplots(3, 2, figsize=(7, 9.6))
for row, col in enumerate(["short_long_ratio", "rna.Mbp", "rna.Prox1"]):
    plot_spatial(cd, q[col], ax=axes[row], cmap="magma" if row == 0 else "viridis",
                 title=col.replace("rna.", ""))
plt.show()

Regions from marker sets of the spot RNA, smoothed over the spot's neighbours (cortex, CA, DG, white matter,
choroid plexus, thalamus; a spot is left `other` when no set stands out), and the paper's statement: the ratio is
higher in cortex, CA and DG than in white matter and the choroid plexus.

In [ ]:
markers = {"white_matter": ["Mbp", "Plp1", "Mog", "Cldn11", "Mal"], "choroid_plexus": ["Ttr", "Kl"],
           "DG": ["Prox1"], "CA": ["Neurod6"], "cortex": ["Satb2", "Cux2", "Rorb", "Fezf2"],
           "thalamus": ["Slc17a6"]}
from scipy.spatial import cKDTree
region = pd.Series("other", index=c.index)
for sec, g in c.groupby("section", observed=True):
    score = pd.DataFrame({r: g[[f"rna.{m}" for m in ms if f"rna.{m}" in g]].mean(axis=1) for r, ms in markers.items()})
    nb = cKDTree(g[["centroid_x", "centroid_y"]].to_numpy()).query_ball_point(g[["centroid_x", "centroid_y"]].to_numpy(), r=2.01)
    sm = np.array([score.to_numpy()[n].mean(0) for n in nb])
    z = (sm - sm.mean(0)) / sm.std(0)
    region[g.index] = np.where(z.max(1) > 0.5, np.array(list(markers))[z.argmax(1)], "other")
c["region"] = region
q = c[c.qc]
print(q.groupby(["section", "region"], observed=True)["short_long_ratio"].median().unstack(0).round(2))
for sec, g in q.groupby("section", observed=True):
    hi = g[g.region.isin(["cortex", "CA", "DG"])].short_long_ratio
    lo = g[g.region.isin(["white_matter", "choroid_plexus"])].short_long_ratio
    u = stats.mannwhitneyu(hi, lo, alternative="greater")
    print(f"{sec}: cortex/CA/DG median {hi.median():.2f} vs white matter/choroid plexus {lo.median():.2f}, "
          f"AUC {u.statistic / (len(hi) * len(lo)):.2f}")

## 3. Barcode stripes

Every array row and every column is barcoded through its own channel: a channel's efficiency adds an offset to
all its spots.  `destripe` fits the row and column offsets together with a smooth surface (cubic B-splines, knots
every 6 spots) and removes only their rough part — the jumps between neighbouring rows / columns; the smooth
pattern (anatomy, and a slow drift that cannot be told apart from it) stays.  The depth of the spots shows the
stripes best.

In [ ]:
g = q[q.section == "mouse_brain_r1"]
rows, cols = g.centroid_y.to_numpy(int), g.centroid_x.to_numpy(int)
ds_ = destripe(g[["log_contacts", "short_long_ratio", "cis_fraction", "frac_2M_12M"]], rows, cols)
fig, axes = plt.subplots(1, 2, figsize=(7, 3.4))
plot_spatial(cd, g["log_contacts"], ax=axes[0], title="log10 contacts")
plot_spatial(cd, ds_["log_contacts"], ax=axes[1], title="destriped")
plt.show()
xy = g[["centroid_x", "centroid_y"]].to_numpy(float)
print("smooth share of the variance, raw -> destriped:",
      dict(zip(ds_.columns, zip(smooth_share(g[ds_.columns], xy).round(2), smooth_share(ds_, xy).round(2)))))

## 4. Spatial tests, raw and adjusted

Per section: Moran's I over the 8 adjacent spots, and SPARK-X (eleven coordinate kernels, Cauchy-combined,
Benjamini–Yekutieli over features) on the destriped values — raw, with the spot's depth, and with depth + the
30 RNA principal components of the store (`cellm["rna_pca"]`).  The share of the smooth spatial variance that
is left after each adjustment is the effect size; with ~6,000 spots every p-value is tiny.  The undigested /
self-ligated share (contacts under 2.5 kb) is a library QC value: a contrast that the RNA should not explain.

In [ ]:
o = cd.cellm["contacts.orientation"].astype(float)
c["frac_lt2k5"] = o[:, :, 0].sum(1) / o.sum((1, 2))       # histogram column 0: d < 2.5 kb
q = c[c.qc]
pcs = pd.DataFrame(np.asarray(cd.cellm["rna_pca"]), index=c.index)
feats = ["short_long_ratio", "frac_2M_12M", "ps_slope_1M_10M", "cis_fraction", "frac_lt2k5"]

def resid(v, X):
    Xi = np.c_[np.ones(len(v)), X]
    b, *_ = np.linalg.lstsq(Xi, v, rcond=None)
    return v - Xi @ b

rows_ = []
for sec, g in q.groupby("section", observed=True):
    xy = g[["centroid_x", "centroid_y"]].to_numpy(float)
    D = destripe(g[feats], g.centroid_y.to_numpy(int), g.centroid_x.to_numpy(int))
    dep = g[["log_contacts"]].to_numpy()
    P = pcs.loc[g.index].to_numpy()
    graph = spatial_neighbors(cd, method="radius", radius=1.5, cells=g.index)
    base = smooth_share(D, xy) * D.var().to_numpy()
    for name, X in (("raw", None), ("depth", dep), ("depth + RNA", np.c_[dep, P])):
        R = D if X is None else pd.DataFrame({k: resid(D[k].to_numpy(), X) for k in feats}, index=g.index)
        I = spatial_autocorr(values=R, graph=graph)["I"]
        left = smooth_share(R, xy) * R.var().to_numpy() / base
        sx = sparkx(D, xy, covariates=X)
        for k in feats:
            rows_.append({"section": sec, "adjusted for": name, "feature": k, "Moran I": I[k],
                          "smooth variance left": left[feats.index(k)], "SPARK-X padj": sx.loc[k, "padj"]})
res = pd.DataFrame(rows_)
res.pivot_table(index=["feature", "adjusted for"], columns="section",
                values=["Moran I", "smooth variance left"]).round(3)

The contact features vary strongly in space (Moran's I 0.5–0.7), and the spot's transcriptome accounts for most
of it: about a tenth of their smooth spatial variance is left after the RNA, while the library QC value keeps
about half.  Is the tenth that is left 3-D genome variation beyond the cells?  An adjustment by noisy covariates
is never complete, so it needs a yardstick: genes held out of the RNA components, adjusted the same way.
`benchmarks/spatial_variable` (stage `calibrate`) does this on the full transcriptome: spatially patterned
held-out genes keep about 30 % of their smooth variance, *more* than the contact features keep.  On these data the
global 3-D genome features follow the cells at least as closely as the cells' other genes do.

Next: the same tests for known structures scored in every spot — boundary insulation and loop strength from
contacts counted in their windows (`uchrom.io.WindowCounts`, `uchrom.fea.window_scores`) and compartment values
(`uc.tl.single_cell_compartments`) — in [spatial variation across a tissue](../schic/spatial.md).